# FINM 32000 — Homework 1

Completed version. Each `FILL THIS IN` from the original template is implemented, with brief math notes.

In [1]:
import plotly.io as pio
pio.renderers.default = 'notebook_connected'

In [2]:
from copy import copy
import numpy as np
import pandas as pd
import plotly.graph_objs as go
from plotly.subplots import make_subplots
from scipy.optimize import bisect, brentq
from scipy.stats import norm

In [3]:
class CallOption:

    def __init__(self, K, T, price=None):
        self.K = K
        self.T = T
        self.price = price

In [4]:
class GBMdynamics:

    def __init__(self, X, r, rGrow, sigma=None):
        self.X = X
        self.r = r
        self.rGrow = rGrow
        self.sigma = sigma

    def update_sigma(self, sigma):
        self.sigma = sigma
        return self

### Black–Scholes engine

The implied-vol solver below brackets the volatility in `[lo, hi]` (the supplied while-loops guarantee the root lies inside) and then calls `scipy.optimize.brentq` on the residual

$$f(\sigma) \;=\; \text{BSpriceCall}(\sigma) - C^{\text{obs}}.$$

In [5]:
class AnalyticEngine:

    def __init__(self):
        pass

    def BSpriceCall(self, dynamics, contract):
        # Ignores contract.price if given, because this function calculates price based on the dynamics.
        # Returns time-0 price.

        F = dynamics.X*np.exp(dynamics.rGrow*contract.T)
        std = dynamics.sigma*np.sqrt(contract.T)
        d1 = np.log(F/contract.K)/std+std/2
        d2 = d1-std
        return np.exp(-dynamics.r*contract.T)*(F*norm.cdf(d1)-contract.K*norm.cdf(d2))

    def BSvega(self, dynamics, contract):
        # Returns time-0 vega
        F = dynamics.X*np.exp(dynamics.rGrow*contract.T)
        std = dynamics.sigma*np.sqrt(contract.T)
        d1 = np.log(F/contract.K)/std+std/2
        return np.exp(-dynamics.r*contract.T)*F*norm.pdf(d1)*np.sqrt(contract.T)

    def IV(self, dynamics, contract):
        # ignores dynamics.sigma; solves for sigma.

        if contract.price is None:
            raise ValueError('Contract price must be given')

        df = np.exp(-dynamics.r*contract.T)
        F = dynamics.X*np.exp(dynamics.rGrow*contract.T)
        lowerbound = np.max([0, df*(F - contract.K)])
        C = contract.price
        if C < lowerbound:
            return np.nan
        if C == lowerbound:
            return 0
        if C >= F*df:
            return np.nan

        dynamics_try = copy(dynamics)
        sigma_try = 0.2
        while self.BSpriceCall(dynamics_try.update_sigma(sigma_try), contract) > C:
            sigma_try /= 2
        while self.BSpriceCall(dynamics_try.update_sigma(sigma_try), contract) < C:
            sigma_try *= 2
        hi = sigma_try
        lo = hi/2

        # Solve BSpriceCall(sigma) = C on [lo, hi] using Brent's method.
        impliedVolatility = brentq(
            lambda s: self.BSpriceCall(dynamics_try.update_sigma(s), contract) - C,
            lo, hi, xtol=1e-12
        )

        return impliedVolatility

## Problem 1

### 1(a) — Inferring the forward price $F_0$

Put–call parity at strike $K$:
$$C_0(K) - P_0(K) \;=\; e^{-rT}(F_0 - K) \;\;\Longrightarrow\;\; F_0 \;=\; K + e^{rT}\bigl(C_0(K)-P_0(K)\bigr).$$

We pick $K^\star$ that minimises $|C_0-P_0|$ (the most ATM strike) to reduce frictions.

In [6]:
options = pd.read_csv('hw1-rut.csv')
options.set_index('Strike', drop=False, inplace=True)

T = 25/365
r = 0.0432

options['CallMid'] = (options.CallBid + options.CallAsk) / 2
options['PutMid']  = (options.PutBid  + options.PutAsk ) / 2

In [7]:
# Calculate the forward price.
# Pick K* minimising |CallMid - PutMid|, then F0 = K* + exp(rT)*(C(K*) - P(K*))
abs_diff = (options.CallMid - options.PutMid).abs()
Kstar = abs_diff.idxmin()
F = Kstar + np.exp(r*T) * (options.loc[Kstar, 'CallMid'] - options.loc[Kstar, 'PutMid'])
print(f"K* (most ATM strike) = {Kstar}")
print(f"CallMid - PutMid at K* = {options.loc[Kstar,'CallMid'] - options.loc[Kstar,'PutMid']:.4f}")
print(f"Inferred forward price F0 = {F:.4f}")

K* (most ATM strike) = 2265
CallMid - PutMid at K* = 0.4000
Inferred forward price F0 = 2265.4012


### 1(b) — Implied-volatility skew

For each strike, transform the (midpoint) put price into an implied call price using parity:
$$C^{\text{impl}}_0(K) \;=\; P_0(K) + e^{-rT}(F_0 - K).$$

At strikes $K>F_0$ the call is OTM, so we use the actual call mid; at $K<F_0$ we use the put-implied call mid. We then invert Black–Scholes for $\sigma$.

In [8]:
# Implied call price obtained from the (midpoint) put price at each strike, via put-call parity.
options['ImpliedCallMid'] = options.PutMid + np.exp(-r*T) * (F - options.Strike)

In [9]:
# Use the OTM option price at each strike:
#   K > F0 -> call is OTM, use CallMid
#   K < F0 -> put  is OTM, use ImpliedCallMid (put-implied call mid)
options['CallOrImpliedCall'] = np.where(options.Strike > F,
                                        options.CallMid,
                                        options.ImpliedCallMid)

In [10]:
hw1analytic = AnalyticEngine()
dynamics = GBMdynamics(X=F, r=r, rGrow=0, sigma=None)

In [11]:
options['IV'] = options.apply(
    lambda row: hw1analytic.IV(dynamics, CallOption(K=row.Strike, T=T, price=row.CallOrImpliedCall)),
    axis=1
)
options[['Strike','CallMid','PutMid','CallOrImpliedCall','IV']].head()

,Strike,CallMid,PutMid,CallOrImpliedCall,IV
Strike,,,,,
1750,1750,515.35,1.200,515.078417,0.451710
1760,1760,505.45,1.250,505.157962,0.445482
1770,1770,495.50,1.300,495.237507,0.439144
1780,1780,485.60,1.375,485.342053,0.433850
1790,1790,475.70,1.450,475.446598,0.428356


In [12]:
fig = go.Figure()

fig.add_trace(go.Scatter(x=options.Strike, y=options.IV, mode='markers', name='Implied Volatility'))

fig.update_layout(title='Implied Volatility vs. Strike',
                  xaxis_title='Strike Price',
                  yaxis_title='Implied Volatility')
fig.show()

## Problem 2

### 2(a) — Polynomial fit in log-moneyness

Define $k:=\log(K/F_0)$ and fit
$$\sigma_p(k) \;=\; \alpha_0 + \alpha_1 k + \alpha_2 k^2 + \alpha_3 k^3 + \alpha_4 k^4$$
in the least-squares sense. `np.polyfit` returns coefficients in **descending** order, which is exactly what `np.poly1d` expects, so we pass them straight through.

In [13]:
k = np.log(options.Strike / F)

mask = options.IV.notna()
coefficients = np.polyfit(k[mask], options.IV[mask], deg=4)
polynomial = np.poly1d(coefficients)

# Report alpha_0 ... alpha_4 (ascending order) for clarity:
alphas = coefficients[::-1]
for i, a in enumerate(alphas):
    print(f"alpha_{i} = {a:.6f}")

alpha_0 = 0.218082
alpha_1 = -0.460467
alpha_2 = 2.361710
alpha_3 = 7.057827
alpha_4 = 20.110443


In [14]:
k_grid = np.linspace(k.min(), k.max(), 100)
y_fit = polynomial(k_grid)

fig = go.Figure()

fig.add_trace(go.Scatter(x=k, y=options.IV, mode='markers', name='Observed midpoint implied volatility'))
fig.add_trace(go.Scatter(x=k_grid, y=y_fit, mode='lines', name='Fitted polynomial parameterization'))

fig.update_layout(title='Implied Volatility vs log-moneyness',
                  xaxis_title='log-moneyness',
                  yaxis_title='Implied Volatility')

fig.show()

### 2(b) — Most overpriced OTM contract (in IV terms)

The contract whose **observed** IV is highest above the fitted polynomial — i.e. the largest positive residual $\sigma^{\text{obs}}(K) - \sigma_p(k)$.

In [15]:
options['IV_fit']   = polynomial(k)
options['IV_resid'] = options.IV - options.IV_fit

overpricedStrike = int(options['IV_resid'].idxmax())
print(f"Most overpriced strike (largest IV residual): {overpricedStrike}")
print(f"  observed IV = {options.loc[overpricedStrike,'IV']:.4f}")
print(f"  fitted   IV = {options.loc[overpricedStrike,'IV_fit']:.4f}")
print(f"  residual    = {options.loc[overpricedStrike,'IV_resid']:.4f}")

Most overpriced strike (largest IV residual): 2570
  observed IV = 0.2254
  fitted   IV = 0.2168
  residual    = 0.0086


### 2(c) — Sensitivity of price to the skew slope $\alpha_1$

Holding $k$ and $\alpha_0,\alpha_2,\alpha_3,\alpha_4$ fixed,
$$\frac{\partial \sigma_p(k)}{\partial \alpha_1} \;=\; k.$$

By the chain rule, the price of any OTM option (call or put) responds to $\alpha_1$ only through the implied vol it gets plugged into Black–Scholes, so
$$\boxed{\;\frac{\partial \text{Price}_0}{\partial \alpha_1}
\;=\; \text{Vega}\cdot k
\;=\; e^{-rT} F_0 \sqrt{T}\, N'(d_1)\,\cdot\,k,\;}$$
with $d_1 = -k/(\sigma_p\sqrt T) + \sigma_p\sqrt T/2$ evaluated at the **fitted** $\sigma_p(k)$.

In [16]:
# The vega calculation for each strike uses the FITTED polynomial implied vol at that strike.
options['vega'] = options.apply(
    lambda row: hw1analytic.BSvega(
        GBMdynamics(X=F, r=r, rGrow=0, sigma=polynomial(np.log(row.Strike / F))),
        CallOption(K=row.Strike, T=T)
    ),
    axis=1
)

In [17]:
# dPrice/d(alpha_1) = vega * k, where k = log(K / F0)
options['skewsensitivity'] = options['vega'] * np.log(options.Strike / F)

In [18]:
import plotly.express as px

fig = px.line(options, x="Strike", y="skewsensitivity", title='Sensitivity to Skew Slope, vs. Strike')
fig.show()

### 2(d) — Biggest gainer / loser when $\alpha_1$ increases

Since $\partial P/\partial \alpha_1 = \text{vega}\cdot k$ and vega is positive for every listed strike, the sensitivity has the same sign as $k=\log(K/F_0)$. The **biggest gainer** sits at high $K$ (positive $k$, large vega) and the **biggest loser** sits at low $K$ (negative $k$, large vega).

In [19]:
whichContractGainsMostFromSlopeIncrease = int(options['skewsensitivity'].idxmax())
print(f"Biggest gainer (alpha_1 up): strike = {whichContractGainsMostFromSlopeIncrease}, "
      f"sensitivity = {options.loc[whichContractGainsMostFromSlopeIncrease,'skewsensitivity']:.4f}")

Biggest gainer (alpha_1 up): strike = 2385, sensitivity = 7.7412


In [20]:
whichContractLosesMostFromSlopeIncrease = int(options['skewsensitivity'].idxmin())
print(f"Biggest loser  (alpha_1 up): strike = {whichContractLosesMostFromSlopeIncrease}, "
      f"sensitivity = {options.loc[whichContractLosesMostFromSlopeIncrease,'skewsensitivity']:.4f}")

Biggest loser  (alpha_1 up): strike = 2100, sensitivity = -9.4291


### Summary of results

| Quantity | Value |
|---|---|
| Forward price $F_0$ | see Problem 1(a) cell |
| Most overpriced strike (IV residual) | see Problem 2(b) cell |
| Biggest gainer as $\alpha_1\uparrow$ | see Problem 2(d) cell |
| Biggest loser as $\alpha_1\uparrow$ | see Problem 2(d) cell |
